# Actividad 1 — Carga y creación de dataset

**Curso:** Proyecto Integrado III - Analítica de Datos
**Dataset:** `t22000t/bike-sharing-tabular` (UCI Bike Sharing Dataset, Capital Bikeshare, Washington D.C., 2011–2012)

## Qué hace este notebook

Coordina las dos clases del proyecto y aplica cinco transformaciones al DataFrame:

```
HFDatasetLoader  →  DataFrame  →  5 transformaciones  →  DatasetSummarizer  →  .txt
```

1. **Carga** con `HFDatasetLoader` (clase en `src/clases_po/hf_loader.py`).
2. **Transforma** el DataFrame (renombrar, seleccionar, filtrar, calcular y agrupar).
3. **Resume** con `DatasetSummarizer` (clase en `src/clases_po/summarizer.py`) y genera el `.txt`.

## 1. Preparar el entorno

Se busca la raíz del proyecto (la carpeta que contiene `src/`) y se agrega `src` a la ruta de importación, para poder usar el paquete `clases_po` sin instalarlo.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "clases_po").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from clases_po import HFDatasetLoader, DatasetSummarizer

print("Raíz del proyecto:", ROOT)

/opt/hostedtoolcache/Python/3.12.15/x64/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Raíz del proyecto: /home/runner/work/proyecto-integrado-iii/proyecto-integrado-iii


## 2. Cargar el dataset desde Hugging Face

`HFDatasetLoader` recibe el identificador del dataset y el split. Internamente llama a `load_dataset()` y devuelve un `pandas.DataFrame`.

In [2]:
loader = HFDatasetLoader(
    dataset_name="t22000t/bike-sharing-tabular",
    split="train",
)

df = loader.load()

print(f"Forma original: {df.shape[0]} filas x {df.shape[1]} columnas")
df.head()

Generating train split:   0%|          | 0/17379 [00:00<?, ? examples/s]

Generating train split: 100%|██████████| 17379/17379 [00:00<00:00, 770740.78 examples/s]

Forma original: 17379 filas x 17 columnas


,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1


**Qué representa cada fila:** una fila es **una hora de operación** del sistema de bicicletas públicas de Washington D.C. — cuántas bicicletas se rentaron en esa hora, con qué clima y en qué momento del calendario.

## 3. Transformaciones

### Transformación 1 — Renombrar columnas al español

Las columnas originales están en inglés y abreviadas (`dteday`, `cnt`, `weathersit`). Se renombran para que el DataFrame sea legible y defendible en el informe.

In [3]:
df1 = df.rename(columns={
    "instant": "indice",
    "dteday": "fecha",
    "season": "temporada",
    "yr": "anio_codigo",
    "mnth": "mes",
    "hr": "hora",
    "holiday": "festivo",
    "weekday": "dia_semana",
    "workingday": "dia_laboral",
    "weathersit": "clima",
    "temp": "temp_norm",
    "atemp": "sensacion_norm",
    "hum": "humedad_norm",
    "windspeed": "viento_norm",
    "casual": "usuarios_casuales",
    "registered": "usuarios_registrados",
    "cnt": "total_rentas",
})

print("Columnas después del renombrado:")
print(list(df1.columns))

Columnas después del renombrado:
['indice', 'fecha', 'temporada', 'anio_codigo', 'mes', 'hora', 'festivo', 'dia_semana', 'dia_laboral', 'clima', 'temp_norm', 'sensacion_norm', 'humedad_norm', 'viento_norm', 'usuarios_casuales', 'usuarios_registrados', 'total_rentas']


### Transformación 2 — Seleccionar un subconjunto de columnas

Se descartan las columnas que no se van a usar en el análisis descriptivo (`indice`, `sensacion_norm`, `viento_norm`, `mes`, `anio_codigo`). Quedan solo las variables que responden la pregunta del proyecto.

In [4]:
columnas = [
    "fecha", "hora", "temporada", "clima", "dia_laboral", "dia_semana",
    "festivo", "temp_norm", "humedad_norm", "usuarios_casuales",
    "usuarios_registrados", "total_rentas",
]

df2 = df1[columnas]

print(f"Forma después de seleccionar: {df2.shape[0]} filas x {df2.shape[1]} columnas")
df2.head(3)

Forma después de seleccionar: 17379 filas x 12 columnas


,fecha,hora,temporada,clima,dia_laboral,dia_semana,festivo,temp_norm,humedad_norm,usuarios_casuales,usuarios_registrados,total_rentas
0,2011-01-01,0,1,1,0,6,0,0.24,0.81,3,13,16
1,2011-01-01,1,1,1,0,6,0,0.22,0.80,8,32,40
2,2011-01-01,2,1,1,0,6,0,0.22,0.80,5,27,32


### Transformación 3 — Filtrar filas con una condición

El proyecto se enfoca en la **movilidad cotidiana**, así que se conservan únicamente los **días laborables**.

Ojo con la definición: en este dataset `dia_laboral` (columna original `workingday`) vale **1 solo si el día no es fin de semana ni festivo**. Por eso el filtro retira **dos tipos de días**:

- los **sábados y domingos** (5.014 horas)
- los **días festivos**, aunque caigan entre semana (500 horas)

En total se descartan 5.514 de las 17.379 horas, es decir el **31,7 %**. La celda muestra el desglose para que no sea un filtro que "hace magia".

> **Consecuencia:** el análisis ya no dice nada sobre el uso recreativo de fin de semana. Es una limitación que debe declararse en el informe.


In [5]:
df3 = df2[df2["dia_laboral"] == 1].copy()

print(f"Filas antes del filtro: {len(df2)}")
print(f"Filas después del filtro: {len(df3)}")
print(f"Filas descartadas: {len(df2) - len(df3)}")

# ¿Qué se retiró exactamente? Se separan las dos causas posibles del descarte.
# En la columna dia_semana: 0 = domingo y 6 = sábado (comprobado con la primera
# fecha del dataset, 2011-01-01, que fue sábado y trae dia_semana = 6).
fin_de_semana = df2["dia_semana"].isin([0, 6])
festivos = df2["festivo"] == 1

print(f"\nHoras en fin de semana (sábado o domingo): {int(fin_de_semana.sum())}")
print(f"Horas en días festivos: {int(festivos.sum())}")

print("\nDías de calendario distintos:")
print(f"  en el DataFrame sin filtrar: {df2['fecha'].nunique()}")
print(f"  conservados (laborables):    {df3['fecha'].nunique()}")
print(f"  retirados:                  {df2['fecha'].nunique() - df3['fecha'].nunique()}")

print("\nRango de fechas conservado:", df3["fecha"].min(), "->", df3["fecha"].max())

Filas antes del filtro: 17379
Filas después del filtro: 11865
Filas descartadas: 5514

Horas en fin de semana (sábado o domingo): 5014
Horas en días festivos: 500

Días de calendario distintos:
  en el DataFrame sin filtrar: 731
  conservados (laborables):    500
  retirados:                  231

Rango de fechas conservado: 2011-01-03 -> 2012-12-31


### Transformación 4 — Crear columnas calculadas

Se agregan cuatro columnas nuevas:

- `fecha` convertida de texto a tipo fecha real (`datetime`)
- `anio` (2011 o 2012) extraído de la fecha — antes era un código 0/1
- `temperatura_c`: la temperatura estaba normalizada (dividida en 41), se devuelve a grados Celsius
- `franja_horaria`: agrupa la hora en cuatro bloques para poder comparar

In [6]:
import pandas as pd

df3["fecha"] = pd.to_datetime(df3["fecha"])

df4 = df3.assign(
    anio=df3["fecha"].dt.year,
    temperatura_c=(df3["temp_norm"] * 41).round(1),
    humedad_pct=(df3["humedad_norm"] * 100).round(1),
    franja_horaria=pd.cut(
        df3["hora"],
        bins=[-1, 5, 11, 17, 23],
        labels=["madrugada", "mañana", "tarde", "noche"],
    ),
)

print("Columnas finales:", list(df4.columns))
print("\nFechas reconstruidas:", df4["fecha"].min().date(), "->", df4["fecha"].max().date())
df4.head(3)

Columnas finales: ['fecha', 'hora', 'temporada', 'clima', 'dia_laboral', 'dia_semana', 'festivo', 'temp_norm', 'humedad_norm', 'usuarios_casuales', 'usuarios_registrados', 'total_rentas', 'anio', 'temperatura_c', 'humedad_pct', 'franja_horaria']

Fechas reconstruidas: 2011-01-03 -> 2012-12-31


,fecha,hora,temporada,clima,dia_laboral,dia_semana,festivo,temp_norm,humedad_norm,usuarios_casuales,usuarios_registrados,total_rentas,anio,temperatura_c,humedad_pct,franja_horaria
47,2011-01-03,0,1,1,1,1,0,0.22,0.44,0,5,5,2011,9.0,44.0,madrugada
48,2011-01-03,1,1,1,1,1,0,0.20,0.44,0,2,2,2011,8.2,44.0,madrugada
49,2011-01-03,4,1,1,1,1,0,0.16,0.47,0,1,1,2011,6.6,47.0,madrugada


### Transformación 5 — Agrupar y agregar

Se resume la demanda promedio por temporada. Esta operación **cambia la forma** del DataFrame (de 11.865 filas a 4), así que es una transformación distinta de las anteriores: agrega, no reorganiza.

In [7]:
por_temporada = (
    df4.groupby("temporada")["total_rentas"]
    .agg(["count", "mean", "sum"])
    .round(2)
    .rename(columns={"count": "horas_registradas",
                     "mean": "promedio_rentas",
                     "sum": "total_rentas"})
)

por_temporada

,horas_registradas,promedio_rentas,total_rentas
temporada,,,
1,2794,119.42,333665
2,3066,205.68,630613
3,3141,238.48,749073
4,2864,202.19,579059


### Comprobación de calidad

Antes de resumir se verifica una regla del dataset: el total debe ser igual a la suma de usuarios casuales y registrados. Si no coincide, hay un problema en los datos.

In [8]:
descuadre = (df4["total_rentas"] != df4["usuarios_casuales"] + df4["usuarios_registrados"]).sum()
print("Filas donde total_rentas != casuales + registrados:", int(descuadre))
print("Valores nulos en el DataFrame transformado:", int(df4.isna().sum().sum()))

Filas donde total_rentas != casuales + registrados: 0
Valores nulos en el DataFrame transformado: 0


## 4. Generar el resumen en `.txt` con `DatasetSummarizer`

Se pasa **el DataFrame transformado** (no el original) a la clase de resumen, y se guarda el archivo dentro del repositorio.

In [9]:
summarizer = DatasetSummarizer(
    dataset_id="t22000t/bike-sharing-tabular",
    split="train",
    nota="DataFrame transformado: columnas renombradas, subconjunto seleccionado, "
         "solo dias laborables, columnas calculadas agregadas.",
)

ruta = summarizer.guardar(df4, ROOT / "EA1" / "resultados" / "resumen_dataset.txt")

print("Archivo generado:", ruta.relative_to(ROOT))

Archivo generado: EA1/resultados/resumen_dataset.txt


## 5. Verificar el archivo generado

Se lee el archivo recién escrito para comprobar que corresponde al DataFrame transformado.

In [10]:
contenido = ruta.read_text(encoding="utf-8")
print(contenido[:1800])

RESUMEN DEL DATAFRAME
Dataset: t22000t/bike-sharing-tabular
Split: train
Generado: 2026-10-24 18:28
Nota: DataFrame transformado: columnas renombradas, subconjunto seleccionado, solo dias laborables, columnas calculadas agregadas.

DIMENSIONES
--------------------------------------------------------------
Filas: 11865
Columnas: 16
Memoria aproximada: 1.41 MB
Filas duplicadas: 0
Valores nulos totales: 0

COLUMNAS Y TIPOS
--------------------------------------------------------------
- fecha: datetime64[us]
- hora: int64
- temporada: int64
- clima: int64
- dia_laboral: int64
- dia_semana: int64
- festivo: int64
- temp_norm: float64
- humedad_norm: float64
- usuarios_casuales: int64
- usuarios_registrados: int64
- total_rentas: int64
- anio: int32
- temperatura_c: float64
- humedad_pct: float64
- franja_horaria: category

VALORES NULOS POR COLUMNA
--------------------------------------------------------------
- fecha: 0 (0.00 %)
- hora: 0 (0.00 %)
- temporada: 0 (0.00 %)
- clima: 0 (0.00 

## 6. Conclusión

- `HFDatasetLoader` obtuvo el dataset de Hugging Face y devolvió un DataFrame de 17.379 filas.
- Se aplicaron **cinco transformaciones**: renombrar, seleccionar columnas, filtrar, crear columnas calculadas y agrupar.
- `DatasetSummarizer` escribió el resumen en `EA1/resultados/resumen_dataset.txt`, en UTF-8, correspondiente al DataFrame transformado (11.865 filas × 16 columnas).
- El archivo `.txt` describe el mismo DataFrame que se muestra en este notebook.